# Backpropagation and The Chain Rule

## 1. Educational Objectives
This notebook breaks down the backpropagation algorithm step-by-step. In our Multilayer Perceptron, backpropagation is how the network learns. After making a prediction (the **forward pass**), the network calculates its error (using **Binary Cross-Entropy Loss**). Backpropagation then works backward from the output layer to the input layer, determining how much each weight and bias in the network contributed to that error.

We use **Gradient Descent** to adjust the weights and minimize the error. To find the gradients (derivatives), we apply the **Chain Rule** of calculus.

### Strict "No-Magic" Constraints
As per our requirements, we are building this entirely from scratch. We will ONLY use `numpy` for matrix operations. No automated differentiation libraries (Autograd, TensorFlow, PyTorch, etc.) are allowed.

## 2. The Chain Rule Matrix Math

During the forward pass, a single layer performs two steps:
1. **Linear combination:** $Z = A_{prev} \cdot W + b$
2. **Activation:** $A = \text{activation}(Z)$

*(Note on notation: mathematical convention uses $A$ for "Activation" rather than $P$ for "Prediction". While the final layer's activation represents the model's prediction, intermediate layers represent internal activations. Therefore, $A_{prev}$ is retained to ensure standard matrix calculus notation).*

To adjust $W$ and $b$, we need to calculate $\frac{\partial E}{\partial W}$ ($dW$) and $\frac{\partial E}{\partial b}$ ($db$). 
Because the error $E$ is at the very end of the network, we must chain derivatives backwards. Assuming we receive the error signal from the layer ahead of us, denoted as $dA_{next}$ (or $\frac{\partial E}{\partial A}$), we calculate four local matrices:

### 1. The Local Error ($dZ$)
#### <span style="color: red;">The first step tells each individual neuron how wrong it was</span>

$$dZ = dA_{next} \odot \text{activation\_derivative}(Z)$$

<b>where</b>

* $dA_{next}$ is the error signal from the next layer 
* $Z$ is the pre-activation output of the current layer (the linear combination before applying the activation function) during the **forward pass**.
* And $\text{activation\_derivative}(Z)$ is the derivative of the activation function with respect to $Z$. 

When we multiply these together, we get $dZ$, which tells us how much each neuron's output contributed to the error.

*Note: $\odot$ is element-wise multiplication (Hadamard product).*
* To **to undestand more in detail** look the following reference: https://app.notion.com/p/jvalenci/GTD-Getting-Things-Done-14f9d52658e08024afb4f480ed354703?p=3799d52658e0808db1f2c82d78b552f1&pm=s

### 2. The Weight Gradient ($dW$)
#### <span style="color: red;">The second step looks at the whole batch to figure out exactly how to modify the ingredients dials ( the weights) that caused the mistake</span>

Next, we determine how the weights contributed to $Z$. Since $Z = A_{prev} W$, the derivative with respect to $W$ involves $A_{prev}$. By the chain rule across a batch of $m$ examples:
$$dW = \frac{1}{m} (A_{prev}^T \cdot dZ)$$

<b>where:</b>
* $A_{prev}^T$ is the transpose of the input activations from the previous layer or we can call it the input features.
* $dZ$ is the local error we just calculated.
* $dW$ is the average gradient of the weights across the batch, which we will use to update the weights during gradient descent.

<details>
<summary><b> The Calculus Breakdown </b></summary>


Here is exactly how this matrix equation maps perfectly to the calculus you already know.

Our goal is to find **$dW$**. In calculus notation, $dW$ is shorthand for $\frac{\partial \text{Loss}}{\partial W}$ (the derivative of the total Error with respect to the Weights).

To find this using the Chain Rule, we need our two pieces: the "Outside" derivative and the "Inside" derivative.

**1. The "Outside" Derivative ($dZ$)**
We already calculated this in the previous step! $dZ$ is just shorthand for $\frac{\partial \text{Loss}}{\partial Z}$. It represents how much the final Error changes based on the output of the neurons.

**2. The "Inside" Derivative ($A_{prev}$)**
This is where your calculus skills shine.
Think back to the forward pass equation for a neuron:


$$Z = A_{prev} \cdot W$$

Let's take the derivative of $Z$ with respect to the variable $W$.

* Treat $A_{prev}$ exactly like a constant number (like a $5$).
* The variable is $W$ (which is technically $W^1$).
* Using the **Power Rule**, the $W$ drops away entirely, leaving just the constant in front!
* Therefore, the derivative of $A_{prev} \cdot W$ is simply **$A_{prev}$**.

**3. Applying the Chain Rule (Multiply them together)**
The Chain Rule says we multiply the outside derivative by the inside derivative:


$$\frac{\partial \text{Loss}}{\partial W} = \text{Inside Derivative} \times \text{Outside Derivative}$$

$$dW = A_{prev} \times dZ$$

### **Why it looks slightly different**

If we were just doing this for one single number, the equation would literally be $dW = A_{prev} \cdot dZ$.

The *only* reason we add the Transpose ($T$) and the averaging factor ($\frac{1}{m}$) is because we are forcing computers to do this exact calculus operation for thousands of examples and thousands of weights all at the exact same time. The core engine driving the math is entirely the Chain Rule!

</details>

*Why the transpose? To align the dimensions between the (# of examples $\times$ # of features) input matrix and the (# of examples $\times$ # of nodes) error matrix $dZ$.*

<details>
<summary><b> The Magic of the Transpose </b></summary>

### **2. The Magic of the Transpose ($A_{prev}^T$)**

The transpose (flipping the matrix on its side) is the most elegant trick in backpropagation. It solves two problems at the exact same time: a structural problem and a mathematical problem.

#### **The Structural Problem (Matching Dimensions)**

Let's look at the shapes of our grids. Let's assume we have a batch of $100$ examples, $3$ input features, and $5$ output neurons.

* **Our Weights ($W$):** We need to know how to update the weights. The weight matrix is shaped **$3 \times 5$** (Inputs $\times$ Outputs). Our resulting $dW$ *must* be this exact same shape!
* **$A_{prev}$ (The Inputs):** Shaped **$100 \times 3$** (Examples $\times$ Inputs).
* **$dZ$ (The Error):** Shaped **$100 \times 5$** (Examples $\times$ Outputs).

If you try to multiply a $100 \times 3$ matrix by a $100 \times 5$ matrix, the rules of linear algebra will throw a massive error. They don't fit together.

#### **The Solution: The Flip**

We transpose $A_{prev}$. Its shape flips from $100 \times 3$ to **$3 \times 100$**.

Now look at what happens when we multiply them:


$$(3 \times 100) \cdot (100 \times 5) \rightarrow \mathbf{3 \times 5}$$

By flipping the inputs on their side, the matrix multiplication perfectly collapses the $100$ examples, leaving us with a beautiful $3 \times 5$ grid of gradients—the exact shape of our weight matrix!

</details>


### 3. The Bias Gradient ($db$)
Since $b$ is added to every example in the batch, its gradient is the sum of $dZ$ across all examples:

$$db = \frac{1}{m} \sum_{i=1}^{m} dZ^{(i)}$$

* $\frac{1}{m}$ is used to average the bias gradient across the batch, ensuring that our updates are stable and not too large.
* $\sum_{i=1}^{m} dZ^{(i)}$ sums the local errors across all examples, giving us the total contribution of the bias to the error.

**In english this is saying:** 
* the $average$ of the $sum$ of the local errors across the batch


At the end of the day, $db$ tells us how much we need to adjust the bias to reduce the error for the entire batch.

### 4. The Upstream Error ($dA_{prev}$)
Finally, we must pass the error backwards so the previous layer can calculate its own $dZ$. Since $Z = A_{prev} W$, the derivative with respect to $A_{prev}$ is $W$:
$$dA_{prev} = dZ \cdot W^T$$
*Why the transpose here? Again, matrix algebra dictates the shapes must align to propagate the (# of nodes) errors backwards into the (# of input features) representation.*



<details>
<summary><b> The Calculus Breakdown </b></summary>

### **The Calculus Proof**

Our goal is to find the upstream error: **$dA_{prev}$**.
In calculus terms, this is $\frac{\partial \text{Loss}}{\partial A_{prev}}$ (the derivative of the Loss with respect to the inputs from the previous layer).

Let's look at our forward pass formula again:


$$Z = A_{prev} \cdot W$$

**1. The "Outside" Derivative ($dZ$)**
Just like every other step in this layer, the error signal coming in from the top is $dZ$.

**2. The "Inside" Derivative (The Power Rule)**
Now we take the derivative of $Z = A_{prev} \cdot W$.

* This time, we are taking the derivative with respect to **$A_{prev}$**.
* Treat **$W$** exactly like a constant number (like a $5$).
* The variable is **$A_{prev}$** (which is technically $A_{prev}^1$).
* Using the **Power Rule**, the variable $A_{prev}$ drops away entirely (it becomes $1$), leaving just the constant in front!
* Therefore, the inside derivative is simply **$W$**.

**3. Applying the Chain Rule**
Multiply the outside by the inside:


$$\text{Outside} \times \text{Inside}$$

$$dZ \times W$$

The $W$ didn't disappear because, in this specific step, the weights were acting as the constant multiplier, not the variable!
</details>

## 3. The Loss Gradient ($dA$)

To kick off the backpropagation process, we must calculate the derivative of the loss function with respect to the final predictions (the output of the last network layer). 

For **Binary Cross-Entropy (BCE)**, the error equation is:
$$E = -\frac{1}{N} \sum [y_n \log(\hat{y}_n) + (1 - y_n) \log(1 - \hat{y}_n)]$$

Taking the derivative with respect to the predicted probabilities ($\hat{y}$, which we denote as $A_{final}$ in our implementation), we get:
$$dA_{final} = \frac{\partial E}{\partial A_{final}} = \frac{1}{N} \left( -\frac{y}{A_{final}} + \frac{1 - y}{1 - A_{final}} \right)$$

This $dA_{final}$ matrix represents how much the final error would change if the predicted probabilities changed. We will pass this backwards into the chain rule to start computing the weights and bias gradients!

In [19]:
import numpy as np
import colorama

def binary_cross_entropy_prime(y_true, y_pred):
    """
    Computes the derivative of binary cross-entropy loss with respect to predictions.
    This kicks off backpropagation by telling the final layer how wrong its predictions were.
    
    Args:
        y_true (np.ndarray): The actual ground truth labels (0 or 1).
        y_pred (np.ndarray): The probabilities predicted by the model (0 to 1).
        
    Returns:
        np.ndarray: The gradient of the loss with respect to the predictions (dA).
    """
    m = y_true.shape[0]  # Number of examples in the batch
    
    # Epsilon to prevent division by zero in log/divisions
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    # Calculate the gradient dA
    # Formula: dA = (1/m) * (-(y/A) + (1-y)/(1-A))
    dA = -(y_true / y_pred) + ((1 - y_true) / (1 - y_pred))
    
    # We average the error over the batch
    dA = dA / m
    return dA

# --- Mock Data to test ---
# Let's pretend we have a batch of 4 examples
y_real = np.array([[1], [0], [1], [0]])
y_predictions = np.array([[0.9], [0.2], [0.1], [0.8]]) # The network was very wrong on the last two!

dA_initial = binary_cross_entropy_prime(y_real, y_predictions)

print("True Labels:\n", y_real.T)
print("Predictions:\n", y_predictions.T)
print("\nInitial Loss Gradient (dA):")
print(dA_initial)
print("\nNotice how the gradients are massive for the wrong predictions (0.1 and 0.8),")
print("pulling them strongly in the correct direction!")

True Labels:
 [[1 0 1 0]]
Predictions:
 [[0.9 0.2 0.1 0.8]]

Initial Loss Gradient (dA):
[[-0.27777778]
 [ 0.3125    ]
 [-2.5       ]
 [ 1.25      ]]

Notice how the gradients are massive for the wrong predictions (0.1 and 0.8),
pulling them strongly in the correct direction!


## 4. Simulating a Layer's Backward Pass

Now that we have the initial error signal ($dA$), let's see how a single `DenseLayer` takes that signal and computes its local gradients ($dZ$, $dW$, $db$) and then generates the error signal for the layer behind it ($dA_{prev}$).

We will mock the state of a single layer as if it had just completed a forward pass using the **Sigmoid** activation function.

In [20]:
def sigmoid_prime(z):
    """
    Derivative of the Sigmoid activation function.
    Formula: sigmoid(z) * (1 - sigmoid(z))
    """
    s = 1 / (1 + np.exp(-z))
    return s * (1 - s)

# --- Mock Layer State (From a Forward Pass) ---
m = 4 # 4 examples in the batch
# A_prev: The input to this layer (output of the previous layer)
# Shape: (4 examples, 3 features/nodes)
A_prev = np.array([
    [0.5, 0.2, 0.1],
    [0.9, 0.1, 0.3],
    [0.4, 0.4, 0.4],
    [0.2, 0.8, 0.6]
])

# W: The current weights of this layer. Shape: (3 input nodes, 2 output nodes)
W_current = np.array([
    [0.1, 0.2],
    [-0.1, 0.4],
    [0.5, -0.2]
])

# Z: The linear combination calculated during the forward pass
# For this mock, we'll just fabricate Z values (Shape: 4x2)
Z_current = np.array([
    [1.2, -0.5],
    [0.8, 0.1],
    [-0.3, 1.5],
    [2.0, -1.0]
])

# dA_next: The error signal from the layer ahead of us (or the loss function).
# Let's use a fabricated error signal matrix shaped (4x2) matching output nodes
dA_next = np.array([
    [-0.5, 0.2],
    [-0.1, 0.4],
    [0.8, -0.3],
    [-1.2, 0.5]
])


In [21]:


print(colorama.Fore.RED + colorama.Style.BRIGHT + "--- Step 1: Calculate Local Error (dZ) ---" + colorama.Style.RESET_ALL)
# 1. dZ = dA_next * activation_derivative(Z)
# This element-wise multiplication tells each neuron how wrong it was before activation!
dZ = dA_next * sigmoid_prime(Z_current)
print("dZ shape:", dZ.shape)
print(dZ)


--- Step 1: Calculate Local Error (dZ) ---
dZ shape: (4, 2)
[[-0.08894722  0.04700074]
 [-0.02139097  0.09975042]
 [ 0.19556665 -0.04474394]
 [-0.1259923   0.09830597]]


In [31]:
print(
    colorama.Fore.RED
    + colorama.Style.BRIGHT
    + "\n--- Step 2 & 3: Calculate Weight and Bias Gradients (dW, db) ---\n"
    + colorama.Style.RESET_ALL
)

# 2. dW = (1/m) * (A_prev.T dot dZ)
# Look at the transpose! A_prev.T flips to (3x4), dotted with dZ (4x2) -> Result is (3x2) matching W_current!

print(
    colorama.Fore.GREEN
    + colorama.Style.BRIGHT
    + f"A_prev shape:{A_prev.shape} "
    + f"dZ shape:{dZ.shape}\n"
    + colorama.Style.RESET_ALL
)

print(
    colorama.Fore.GREEN
    + colorama.Style.BRIGHT
    + f"A_prev.T shape:{A_prev.T.shape} "
    + f"dZ shape:{dZ.shape}\n"
    + colorama.Style.RESET_ALL
)

dW = (1 / m) * np.dot(A_prev.T, dZ)

print(
    colorama.Fore.GREEN
    + colorama.Style.BRIGHT
    + "This gives as a result a (3,2) matrix for dw\n"
    + colorama.Style.RESET_ALL
)

# 3. db = (1/m) * sum(dZ across batch)
# Keep dimensions ensures db is shape (1, 2)
db = (1 / m) * np.sum(dZ, axis=0, keepdims=True)
print("dW shape:", dW.shape, "(Matches W)")
print(dW)
print(
    colorama.Fore.RED
    + colorama.Style.BRIGHT
    + "----------------------------------------"
    + colorama.Style.RESET_ALL
)

print("db shape:", db.shape)
print(db)


--- Step 2 & 3: Calculate Weight and Bias Gradients (dW, db) ---

A_prev shape:(4, 3) dZ shape:(4, 2)

A_prev.T shape:(3, 4) dZ shape:(4, 2)

This gives as a result a (3,2) matrix for dw

dW shape: (3, 2) (Matches W)
[[-0.00267432  0.02875984]
 [-0.01062393  0.0200306 ]
 [-0.00317018  0.0189278 ]]
----------------------------------------
db shape: (1, 2)
[[-0.01019096  0.0500783 ]]


In [32]:
print(
    colorama.Fore.RED
    + colorama.Style.BRIGHT
    + "\n--- Step 4: Calculate the error that we are going to pass to the previous Layer (dA_prev) ---"
    + colorama.Style.RESET_ALL
)

# 4. dA_prev = dZ dot W_current.T
# dZ is (4x2), W is (3x2). We transpose W to (2x3).
# Matrix multiplication gives (4x2) dot (2x3) -> (4x3), perfectly matching A_prev!
dA_prev = np.dot(dZ, W_current.T)

print(
    colorama.Fore.GREEN
    + colorama.Style.BRIGHT
    + f"dA_prev shape:{dA_prev.shape} (Matches A_prev)"
    + colorama.Style.RESET_ALL
)
print(dA_prev)


--- Step 4: Calculate the error that we are going to pass to the previous Layer (dA_prev) ---
dA_prev shape:(4, 3) (Matches A_prev)
[[ 0.00050543  0.02769502 -0.05387376]
 [ 0.01781099  0.04203926 -0.03064557]
 [ 0.01060788 -0.03745424  0.10673211]
 [ 0.00706196  0.05192162 -0.08265734]]
